In [1]:

df_medicaltest = spark.read.table("Bronze_LH.dbo.bronze_medicaltests")
display(df_medicaltest)


StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b94e3b2f-9fd9-4aab-96ea-7cb109edce64)

#### Data Profileing and Cleaning--> Mediacal Test


In [2]:
df_medicaltest.printSchema()
print("Total_Rows :",df_medicaltest.count())
print("Total_Columns :",df_medicaltest.columns)

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 4, Finished, Available, Finished, False)

root
 |-- TestID: integer (nullable = true)
 |-- PatientID: integer (nullable = true)
 |-- TestType: string (nullable = true)
 |-- TestResult: string (nullable = true)
 |-- TestDate: timestamp (nullable = true)

Total_Rows : 10021
Total_Columns : ['TestID', 'PatientID', 'TestType', 'TestResult', 'TestDate']


In [3]:
## Null profiling 

from pyspark.sql.functions import col , when , trim , lower , desc , count

null_profile=[]

for c in df_medicaltest.columns:
    null_count = df_medicaltest.filter(col(c).isNull()).count()
    null_profile.append((c,null_count))

null_profile_medical_test = spark.createDataFrame(null_profile,["columns","null_count"])
display(null_profile_medical_test)

## Blank Profiling

blank_profile=[]

for c in df_medicaltest.columns:
    blank_count= df_medicaltest.filter(trim(col(c))=="").count()
    blank_profile.append((c,blank_count))

blank_profile_medical_test = spark.createDataFrame(blank_profile,["columns","blank_count"])
display(blank_profile_medical_test)


## Categorical_col

cat_col=["TestType","TestResult"]

for c in cat_col:
    print(f"\n========{c}=======")
    df_medicaltest.groupBy(c).count().orderBy(desc("count")).show(truncate=False)



StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 434b86fd-3442-49a5-a412-86892b48ea34)

SynapseWidget(Synapse.DataFrame, 34ed2f7f-1116-4779-9cfa-e85bce9b3d87)


========TestType=======
+----------------------+-----+
|TestType              |count|
+----------------------+-----+
|Liver Function Test   |975  |
|CT Scan               |972  |
|Urine Test            |938  |
|X-Ray                 |933  |
|Kidney Function Test  |927  |
|Blood Test            |919  |
|ECG                   |916  |
|MRI                   |902  |
|Cholesterol Test      |893  |
|Blood Sugar Test      |866  |
|UnknownTest           |185  |
|NULL                  |169  |
|urine test            |28   |
| Kidney Function Test |28   |
|blood test            |28   |
|ecg                   |27   |
|kidney function test  |25   |
| CT Scan              |25   |
| MRI                  |24   |
| Blood Test           |22   |
+----------------------+-----+
only showing top 20 rows


========TestResult=======
+-------------+-----+
|TestResult   |count|
+-------------+-----+
|Abnormal     |3175 |
|Pending      |3147 |
|Normal       |3122 |
|NULL         |139  |
|InvalidResult|136  |
|n

In [4]:
## Checking and removing Duplicate keys

dups = df_medicaltest.groupBy("TestID").count().filter(col("count")>1).show()

df_medicaltest_clean=df_medicaltest.dropDuplicates()

print("before:",df_medicaltest.count())
print("after :",df_medicaltest_clean.count())

df_medicaltest_clean.groupBy("TestID").count().filter(col("count")>1).show()


StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 6, Finished, Available, Finished, False)

+------+-----+
|TestID|count|
+------+-----+
|  4000|    2|
|  1005|    2|
|  1008|    2|
|   500|    2|
|   650|    2|
|  1002|    2|
|   100|    2|
|   648|    2|
|  1000|    2|
|   649|    2|
|  1001|    2|
|  1006|    2|
|  1007|    2|
|  1003|    2|
|  6000|    2|
|   651|    2|
|  1004|    2|
|  2000|    2|
|   652|    2|
|  8000|    2|
+------+-----+
only showing top 20 rows

before: 10021
after : 10013
+------+-----+
|TestID|count|
+------+-----+
|  1005|    2|
|  1008|    2|
|   650|    2|
|  1002|    2|
|   648|    2|
|   649|    2|
|  1001|    2|
|  1006|    2|
|  1007|    2|
|  1003|    2|
|   651|    2|
|  1004|    2|
|   652|    2|
+------+-----+



In [5]:
silver_medicaltest = df_medicaltest_clean
silver_medicaltest.printSchema()

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 7, Finished, Available, Finished, False)

root
 |-- TestID: integer (nullable = true)
 |-- PatientID: integer (nullable = true)
 |-- TestType: string (nullable = true)
 |-- TestResult: string (nullable = true)
 |-- TestDate: timestamp (nullable = true)



In [6]:
## String Columns
from pyspark.sql.functions import trim , initcap , when , count , lower

str_col =["TestType","TestResult"]

for c in str_col:
    silver_medicaltest=silver_medicaltest.withColumn(c,initcap(trim(lower(col(c)))))

display(silver_medicaltest)

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cbcf3970-9754-4c5c-9607-9ab22d825855)

In [7]:
silver_medicaltest.groupBy("TestType").count().show()
silver_medicaltest.groupBy("TestResult").count().show()

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 9, Finished, Available, Finished, False)

+--------------------+-----+
|            TestType|count|
+--------------------+-----+
| Liver Function Test| 1013|
|                NULL|  169|
|                 Ecg|  954|
|          Blood Test|  970|
|Kidney Function Test|  977|
|    Cholesterol Test|  934|
|                 Mri|  941|
|          Urine Test|  981|
|             Ct Scan| 1015|
|               X-ray|  971|
|         Unknowntest|  185|
|    Blood Sugar Test|  903|
+--------------------+-----+

+-------------+-----+
|   TestResult|count|
+-------------+-----+
|         NULL|  139|
|Invalidresult|  136|
|      Pending| 3244|
|     Abnormal| 3264|
|       Normal| 3230|
+-------------+-----+



In [8]:
from pyspark.sql.functions import concat_ws , lit , col ,when

silver_medicaltest=silver_medicaltest.withColumn(
    "DataQualityReason",

    concat_ws(
        ",",

        when(
            col("PatientID").isNull(),lit("Missing_PatientID")
        ),
        when(
            col("TestType").isNull(),lit("Missing TestType")
        ).when(
            col("TestType")=="Unknowntest",lit("Invalid Test Type")
        ),
        when(
            col("TestResult").isNull(),
            lit("Missing TestResult")
        ).when(
            col("TestResult") == "Invalidresult",
            lit("Invalid TestResult")
        ),
        when(
            col("TestDate").isNull(),
            lit("Missing TestDate")

    )
)
)

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 10, Finished, Available, Finished, False)

In [9]:
from pyspark.sql.functions import col

silver_medicaltest_valid = silver_medicaltest.filter(
    col("DataQualityReason") == ""
)

silver_medicaltest_review = silver_medicaltest.filter(
    col("DataQualityReason") != ""
)


StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 11, Finished, Available, Finished, False)

In [10]:
silver_medicaltest_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_medicaltest_valid")
    

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 12, Finished, Available, Finished, False)

In [11]:
silver_medicaltest_review.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_medicaltest_review")

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 13, Finished, Available, Finished, False)

In [12]:
print("Total records:", silver_medicaltest.count())
print("Valid records:", silver_medicaltest_valid.count())
print("Review records:", silver_medicaltest_review.count())

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 14, Finished, Available, Finished, False)

Total records: 10013
Valid records: 9046
Review records: 967


### Cross Table Validation


In [13]:
patient_valid = spark.table("silver_patients")
patient_review = spark.table("silver_patients_review")

medicaltest_valid = spark.table("silver_medicaltest_valid")
medicaltest_review = spark.table("silver_medicaltest_review")

all_patient_id = (
    patient_valid.select("PatientID")
    .union(
        patient_review.select("PatientID")
    ).distinct()
)

print(f"Total Patient Id:",all_patient_id.count())

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 15, Finished, Available, Finished, False)

Total Patient Id: 992


In [14]:

from pyspark.sql.functions import lit

orphan_medicaltest= medicaltest_valid.join(
    all_patient_id,on="PatientID",how="left_anti"
)
print("Orphan medical test:",orphan_medicaltest.count())

orphan_medicaltest.show()



orphan_medicaltest = orphan_medicaltest.withColumn(
    "DataQualityReason",
    lit("Invalid PatientID - Patient not found")
)

medicaltest_valid_updated = medicaltest_valid.join(
    orphan_medicaltest.select("TestID"),
    on="TestID",
    how="left_anti"
)

medicaltest_review_updated = medicaltest_review.unionByName(
    orphan_medicaltest,
    allowMissingColumns=True
)

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 16, Finished, Available, Finished, False)

Orphan medical test: 324
+---------+------+--------------------+----------+-------------------+-----------------+
|PatientID|TestID|            TestType|TestResult|           TestDate|DataQualityReason|
+---------+------+--------------------+----------+-------------------+-----------------+
|     1013|  6882|    Blood Sugar Test|  Abnormal|2026-04-22 15:54:31|                 |
|     1031|  2886|                 Mri|    Normal|2025-03-15 02:16:28|                 |
|     1097|  4847|    Cholesterol Test|    Normal|2025-07-07 15:43:28|                 |
|     1013|  7326|Kidney Function Test|  Abnormal|2026-04-14 21:39:01|                 |
|     1043|  5550|    Blood Sugar Test|   Pending|2025-07-10 05:33:57|                 |
|     1200|  7104|Kidney Function Test|  Abnormal|2026-05-16 15:16:18|                 |
|      452|  7610| Liver Function Test|  Abnormal|2025-05-21 16:57:10|                 |
|     1021|   111|          Urine Test|  Abnormal|2025-10-12 13:16:53|               

In [15]:
print("Original Valid:", medicaltest_valid.count())
print("Orphans:", orphan_medicaltest.count())
print("Updated Valid:", medicaltest_valid_updated.count())
print("Updated Review:", medicaltest_review_updated.count())

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 17, Finished, Available, Finished, False)

Original Valid: 9046
Orphans: 324
Updated Valid: 8721
Updated Review: 1291


In [16]:
medicaltest_valid_updated.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_medicaltest_valid")

medicaltest_review_updated.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_medicaltest_review")

StatementMeta(, b2c2fe5e-8f9c-46c9-bf40-b8e43f6b21c4, 18, Finished, Available, Finished, False)